# Phase 8.5: FLUX ABC School — Teacher-Guided Curriculum

**Goal**: Teach FLUX to generate coherent English through:
- Gemini teacher scoring & corrections
- Surprise-based thermodynamic learning
- Grade/subject curriculum (K-6)
- Episodic memory growth (74 → 500+ entries)

**Prerequisites**:
- `Flux-beta.flx` (Phase 8 trained model)
- Kaggle secrets: `HF_TOKEN`, `GEMINI_API_KEY`

**Expected Runtime**: ~3-4 hours on T4 GPU

## Cell 1: Clone Repo & Install Dependencies

In [ ]:
%%time
import os
from pathlib import Path

REPO_URL = 'https://github.com/Unseengap/FLUX.git'
ROOT = Path('/content/FLUX')

if ROOT.exists():
    print('  Pulling latest...')
    !cd {ROOT} && git pull
else:
    print('  Cloning repo...')
    !git clone {REPO_URL} {ROOT}

os.chdir(ROOT)
print(f'  Working dir: {os.getcwd()}')

# Install dependencies
!pip install -q -e . 2>/dev/null || pip install -q -r requirements.txt
!pip install -q google-generativeai

print('  ✓ Dependencies installed')

## Cell 2: Setup Paths & Logger

In [ ]:
import sys
from pathlib import Path

ROOT = Path('/content/FLUX')
ROOT_DIR = ROOT  # Alias for compatibility
PHASES_DIR = ROOT / 'phases'
CHECKPOINT_DIR = ROOT / 'checkpoints'
CHECKPOINT_DIR.mkdir(exist_ok=True)

# Add all phase paths
for i in range(1, 9):
    p = PHASES_DIR / f'phase{i}'
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Add phase8_5
p85 = PHASES_DIR / 'phase8_5'
if str(p85) not in sys.path:
    sys.path.insert(0, str(p85)) 

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from flux_utils import PhaseLogger, get_device

log = PhaseLogger(phase=8)  # Log to phase8 (8.5 is extension)
log.separator("Phase 8.5: FLUX ABC School")

print('  ✓ Paths configured')

## Cell 3: Hardware & Secrets

In [ ]:
import torch
import os

log.cell_start("Cell 3 — Hardware & Secrets")

device = get_device()
print(f'  Device: {device}')

if device == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  GPU: {gpu_name} ({vram:.1f} GB)')

# Load secrets (Kaggle → Colab → env fallback)
hf_token = None
gemini_key = None

# Try Kaggle secrets
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret('HF_TOKEN')
    gemini_key = secrets.get_secret('GEMINI_API_KEY')
    print('  ✓ Kaggle secrets loaded')
except:
    pass

# Try Colab secrets
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
        gemini_key = userdata.get('GEMINI_API_KEY')
        print('  ✓ Colab secrets loaded')
    except:
        pass

# Fall back to env vars
if not hf_token:
    hf_token = os.environ.get('HF_TOKEN')
    gemini_key = os.environ.get('GEMINI_API_KEY')
    if hf_token:
        print('  ✓ Environment variables loaded')

if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print('  ✓ HF_TOKEN set')
else:
    print('  ⚠ HF_TOKEN not found')

if gemini_key:
    os.environ['GEMINI_API_KEY'] = gemini_key
    print('  ✓ GEMINI_API_KEY set')
else:
    print('  ⚠ GEMINI_API_KEY not found (will use heuristic scoring)')

log.cell_end("Cell 3 — Hardware & Secrets", "PASS")

## Cell 4: Download Flux-beta.flx

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path

log.cell_start("Cell 4 — Download Flux-beta.flx")

flx_path = CHECKPOINT_DIR / 'Flux-beta.flx'

if flx_path.exists():
    size_mb = flx_path.stat().st_size / 1e6
    print(f'  ✓ Flux-beta.flx exists ({size_mb:.1f} MB)')
else:
    print('  Downloading from HuggingFace...')
    try:
        downloaded = hf_hub_download(
            repo_id='UnseenGAP/FLUX',
            filename='checkpoints/Flux-beta.flx',
            local_dir=str(ROOT_DIR),
            token=hf_token,
        )
        size_mb = Path(downloaded).stat().st_size / 1e6
        print(f'  ✓ Downloaded ({size_mb:.1f} MB)')
    except Exception as e:
        print(f'  ⚠ Download failed: {e}')
        print('  Trying checkpoints/phase8.phase.pt fallback...')
        try:
            downloaded = hf_hub_download(
                repo_id='UnseenGAP/FLUX',
                filename='checkpoints/phase8.phase.pt',
                local_dir=str(ROOT_DIR),
                token=hf_token,
            )
            print(f'  ✓ Fallback downloaded')
            flx_path = CHECKPOINT_DIR / 'phase8.phase.pt'  # Use .pt as input
        except Exception as e2:
            print(f'  ✗ Fallback also failed: {e2}')
            raise RuntimeError("Cannot proceed without checkpoint")

log.cell_end("Cell 4 — Download Flux-beta.flx", "PASS")

## Cell 5: Load FLUXLarge from .flx

In [ ]:
log.cell_start("Cell 5 — Load FLUXLarge from .flx")

from flx_loader import load_flux_from_flx, verify_flx

# Verify first
if flx_path.suffix == '.flx':
    success, msg = verify_flx(flx_path)
    print(f'  Verify: {msg}')
    if not success:
        raise ValueError(f".flx verification failed: {msg}")
    
    # Load from .flx
    model = load_flux_from_flx(flx_path, device=device, verbose=True)
else:
    # Fallback: load from .pt
    from flux_large import FLUXLarge
    model = FLUXLarge.from_phase8_checkpoint(device=device)

# Get stats
stats = model.get_stats()
print(f'\n  Model ready:')
print(f'    Params: {stats.total_params:,}')
print(f'    Episodic entries: {stats.episodic_entries}')
print(f'    Field energy: {stats.field_energy:.2f}')
print(f'    Temperature: {stats.temperature:.4f}')

log.metric('initial_params', str(stats.total_params))
log.metric('initial_episodic', str(stats.episodic_entries))
log.cell_end("Cell 5 — Load FLUXLarge from .flx", "PASS")

## Cell 6: Initialize Teacher & Corrector

In [ ]:
log.cell_start("Cell 6 — Initialize Teacher & Corrector")

from gemini_teacher import GeminiTeacher
from surprise_correction import SurpriseCorrector

# Create teacher
teacher = GeminiTeacher(verbose=True)

if teacher.is_available:
    print('  ✓ Gemini teacher ready')
    log.success('Gemini teacher connected')
else:
    print('  ⚠ Gemini API not available — using heuristic scoring')
    log.warning('Gemini unavailable, using heuristics')

# Create corrector
corrector = SurpriseCorrector(
    model=model,
    surprise_threshold=0.3,
    verbose=True,
)
print('  ✓ Surprise corrector ready')

# Quick test
print('\n  Quick generation test...')
test_output = model.generate("The meaning of life is", max_length=40)
print(f'    "{test_output[:80]}..."')

log.cell_end("Cell 6 — Initialize Teacher & Corrector", "PASS")

## Cell 7: Load OpenWebText for Grade 6

In [ ]:
log.cell_start("Cell 7 — Load OpenWebText")

from train_openwebtext import load_openwebtext_subset

print('  Loading OpenWebText (1000 docs)...')
owt_texts = load_openwebtext_subset(max_docs=1000)
print(f'  ✓ {len(owt_texts)} documents loaded')

# Sample
sample = owt_texts[0][:100] if owt_texts else ''
print(f'  Sample: "{sample}..."')

log.cell_end("Cell 7 — Load OpenWebText", "PASS")

## Cell 8: Run Curriculum School (Main Training)

In [ ]:
log.cell_start("Cell 8 — Curriculum School Training")

from curriculum_school import CurriculumSchool
import time

# Create school
school = CurriculumSchool(
    model=model,
    teacher=teacher,
    log=log,
    openwebtext_texts=owt_texts,
    verbose=True,
)

# Run curriculum
print('\n🏫 Starting school...')
t0 = time.time()

result = school.run_school(
    start_grade=0,  # Kindergarten
    max_grade=6,    # Through Grade 6
)

elapsed = time.time() - t0

print(f'\n  Training complete in {elapsed/60:.1f} minutes')
log.metric('school_time_minutes', f'{elapsed/60:.1f}')
log.metric('grades_completed', str(result.grades_completed))
log.metric('final_episodic', str(result.final_episodic_entries))

log.cell_end("Cell 8 — Curriculum School Training", "PASS")

## Cell 9: Save Phase 8.5 Checkpoint

In [ ]:
log.cell_start("Cell 9 — Save Checkpoint")

import torch
from datetime import datetime

# Build checkpoint
stats = model.get_stats()

checkpoint = {
    'phase': 8.5,
    'timestamp': datetime.now().isoformat(),
    'config': model.config,
    'learning_steps': model._learning_steps,
    
    # School results
    'school_result': {
        'grades_completed': result.grades_completed,
        'graduated': result.graduated,
        'final_episodic': result.final_episodic_entries,
    },
    
    # Component states
    'cse_state_dict': model.cse.state_dict(),
    'field_state_dict': model.field.state_dict(),
    'gr_state': model.gr.save_state(),
    'tl_state': model.tl.save_state(),
    'cgn_state': model.cgn.save_state(),
    'causal_graph_state': model.causal_graph.save_state(),
    'working_memory_state': model.working_memory.state_dict_memory(),
    'episodic_memory_state': model.episodic_memory.save_state(),
    'semantic_memory_state': model.semantic_memory.save_state(),
    'router_state': model.memory_router.save_state(),
    'wave_to_field_state': model.wave_to_field.state_dict(),
    'field_to_wave_state': model.field_to_wave.state_dict(),
    'output_head_state': model.output_head.state_dict(),
    'decoder_state_dict': model.decoder.state_dict(),
    
    'metrics': {
        'total_params': stats.total_params,
        'episodic_entries': stats.episodic_entries,
        'field_energy': stats.field_energy,
        'temperature': stats.temperature,
    },
}

# Save
ckpt_path = CHECKPOINT_DIR / 'phase8_5.phase.pt'
torch.save(checkpoint, str(ckpt_path))
size_mb = ckpt_path.stat().st_size / 1e6
print(f'  ✓ Checkpoint saved: {ckpt_path} ({size_mb:.1f} MB)')

log.success(f'Checkpoint saved: {size_mb:.1f} MB')
log.cell_end("Cell 9 — Save Checkpoint", "PASS")

## Cell 10: Demo — Before vs After Generation

In [ ]:
log.cell_start("Cell 10 — Demo: Generation Quality")

prompts = [
    "The meaning of life is",
    "In the future, we will",
    "Science has shown that",
    "The most important thing is",
    "When you think about it,",
]

print('\n  📝 Generation Samples (Post-School):')
print('  ' + '─' * 60)

for prompt in prompts:
    output = model.generate(prompt, max_length=60, temperature=0.7)
    continuation = output[len(prompt):]
    print(f'\n  Prompt: "{prompt}"')
    print(f'  Output: "{continuation[:80]}..."')

print('\n  ' + '─' * 60)
print(f'  Final episodic entries: {model.episodic_memory.size}')

log.cell_end("Cell 10 — Demo: Generation Quality", "PASS")

## Cell 11: Upload to HuggingFace Hub

In [ ]:
log.cell_start("Cell 11 — Upload to HuggingFace")

from huggingface_hub import HfApi

if hf_token:
    try:
        api = HfApi()
        
        # Upload checkpoint
        api.upload_file(
            path_or_fileobj=str(ckpt_path),
            path_in_repo='phase8_5.phase.pt',
            repo_id='UnseenGAP/FLUX',
            token=hf_token,
        )
        print('  ✓ Checkpoint uploaded to HuggingFace')
        log.success('Uploaded to HuggingFace')
    except Exception as e:
        print(f'  ⚠ Upload failed: {e}')
        log.warning(f'Upload failed: {e}')
else:
    print('  ⚠ No HF_TOKEN — skipping upload')

log.cell_end("Cell 11 — Upload to HuggingFace", "PASS")

## Cell 12: Summary & Stats

In [ ]:
log.cell_start("Cell 12 — Summary")

stats = model.get_stats()
corrector_stats = school.corrector.get_stats()
teacher_stats = teacher.get_stats()

print('\n' + '═' * 60)
print('  📊 PHASE 8.5 TRAINING SUMMARY')
print('═' * 60)

print(f'\n  Model:')
print(f'    Parameters: {stats.total_params:,}')
print(f'    Field energy: {stats.field_energy:.2f}')
print(f'    Temperature: {stats.temperature:.4f}')

print(f'\n  School:')
print(f'    Grades completed: {result.grades_completed}/7')
print(f'    Graduated: {"✓" if result.graduated else "✗"}')
print(f'    Training time: {result.total_time_seconds/60:.1f} min')

print(f'\n  Memory Growth:')
print(f'    Episodic entries: {result.final_episodic_entries}')
print(f'    Episodic writes during training: {corrector_stats["episodic_writes"]}')

print(f'\n  Corrections:')
print(f'    Total: {corrector_stats["total_corrections"]}')
print(f'    Avg surprise: {corrector_stats["avg_surprise"]:.3f}')
print(f'    Write rate: {corrector_stats["write_rate"]:.2%}')

print(f'\n  Teacher:')
print(f'    API calls: {teacher_stats["calls"]}')
print(f'    Success rate: {teacher_stats["success_rate"]:.2%}')

print('\n' + '═' * 60)

log.cell_end("Cell 12 — Summary", "PASS")
log.separator("Phase 8.5 Complete")